# Assignment 4 — N-gram Language Models with Laplace Smoothing

**Language:** Telugu (tokenized sentences from Assignment 1)

This notebook answers both parts of the lab:

1. **Build four language models** (unigram, bigram, trigram, quadrigram) on at least **1,000,000 sentences**, split into **1,000 development**, **1,000 test**, and **the rest training**.
2. **Apply Add-one / Laplace smoothing** using

$$P_{\text{Laplace}}(w_n \mid w_1,\dots,w_{n-1}) = \frac{C(w_1,\dots,w_n)+1}{C(w_1,\dots,w_{n-1})+V}$$

and evaluate the models with **perplexity** on the development and test sets.


## 0. Setup and imports

In [ ]:
import os
import math
import time
import random
from collections import Counter, defaultdict

import pandas as pd

print("Setup complete.")

## 1. Build four n-gram language models

### 1.1 Load tokenized Telugu data (Assignment 1)

The assignment requires **at least 1,000,000 sentences** from the language data tokenized in Assignment 1. We load the first 1,000,000 tokenized IndicCorp Telugu sentences from the parquet file produced in Assignment 1.


In [ ]:
data_path = os.path.join(
    "..", "Assignment1", "output", "indiccorp", "indiccorp_te_tokenized.parquet"
)

df = pd.read_parquet(data_path)
df_subset = df.head(1_000_000)

all_sents = []
for text in df_subset["tokenized_sentence"]:
    if isinstance(text, str):
        tokens = text.split()
        if tokens:
            all_sents.append(tokens)

print(f"Total sentences loaded: {len(all_sents)}")
print("Example sentence:", all_sents[0] if all_sents else None)

### 1.2 Train / development / test split

Per the assignment:

- **Test set:** 1,000 sentences  
- **Development set:** 1,000 sentences  
- **Training set:** all remaining sentences  

Sentences are shuffled with a fixed seed so the split is reproducible.


In [ ]:
random.seed(42)
random.shuffle(all_sents)

test_set = all_sents[:1000]
dev_set = all_sents[1000:2000]
train_set = all_sents[2000:]

print(f"Train sentences: {len(train_set)}")
print(f"Dev sentences:   {len(dev_set)}")
print(f"Test sentences:  {len(test_set)}")
assert len(dev_set) == 1000 and len(test_set) == 1000
assert len(train_set) == len(all_sents) - 2000

### 1.3 Vocabulary and out-of-vocabulary (OOV) handling

The vocabulary is built **only from the training set**. Words in development/test that never appeared in training are mapped to `<UNK>`. Start-of-sentence `<s>` and end-of-sentence `</s>` markers are added so higher-order models have left context at sentence boundaries.


In [ ]:
vocab = set()
for s in train_set:
    vocab.update(s)

vocab.add("<UNK>")
vocab.add("<s>")
vocab.add("</s>")
V = len(vocab)

print(f"Vocabulary size (V): {V:,}")


def replace_unk(sentence):
    """Map any word not seen in training to <UNK>."""
    return [w if w in vocab else "<UNK>" for w in sentence]


def pad(sentence, n):
    """Add (n-1) start markers and one end marker."""
    return ["<s>"] * (n - 1) + sentence + ["</s>"]

### 1.4 Count n-grams for unigram, bigram, trigram, and quadrigram models

For each order $n \in \{1,2,3,4\}$ we store:

- `ngram_counts`: $C(w_1,\dots,w_n)$
- `context_counts`: $C(w_1,\dots,w_{n-1})$ (for unigrams the context is empty and the count is the total number of training tokens)


In [ ]:
def build_ngram_counts(sentences, n):
    ngram_counts = Counter()
    context_counts = Counter()
    for s in sentences:
        padded = pad(replace_unk(s), n)
        for i in range(n - 1, len(padded)):
            ngram = tuple(padded[i - n + 1 : i + 1])
            context = ngram[:-1]
            ngram_counts[ngram] += 1
            context_counts[context] += 1
    return ngram_counts, context_counts


models = {}  # n -> (ngram_counts, context_counts, context_to_words)
for n in [1, 2, 3, 4]:
    t0 = time.time()
    ngc, cc = build_ngram_counts(train_set, n)
    context_to_words = defaultdict(set)
    for ngram in ngc:
        context_to_words[ngram[:-1]].add(ngram[-1])
    models[n] = (ngc, cc, context_to_words)
    name = {1: "Unigram", 2: "Bigram", 3: "Trigram", 4: "Quadrigram"}[n]
    print(
        f"{name:10s} (n={n}): {len(ngc):>11,} unique n-grams, "
        f"{len(cc):>11,} unique contexts  (built in {time.time() - t0:.1f}s)"
    )

# Cached once: unigram denominator is total training tokens
UNIGRAM_TOTAL_TOKENS = sum(models[1][0].values())
print(f"\nTotal training tokens (unigram N): {UNIGRAM_TOTAL_TOKENS:,}")

## 2. Apply Add-one / Laplace smoothing

Use the assignment formula:

$$P_{\text{Laplace}}(w_n \mid w_1,\dots,w_{n-1}) = \frac{C(w_1,\dots,w_n)+1}{C(w_1,\dots,w_{n-1})+V}$$

For the **unigram** model there is no context, so the denominator is $N + V$, where $N$ is the number of training tokens.


In [ ]:
def laplace_prob(ngram, n):
    """Add-one smoothed P(last word | context)."""
    ngc, cc, _ = models[n]
    if n == 1:
        return (ngc[ngram] + 1) / (UNIGRAM_TOTAL_TOKENS + V)
    context = ngram[:-1]
    return (ngc[ngram] + 1) / (cc[context] + V)


# Sanity check on a frequent Telugu unigram from training
most_common_word = models[1][0].most_common(1)[0][0][0]
print(
    f'P("{most_common_word}") under Laplace unigram = '
    f"{laplace_prob((most_common_word,), 1):.8f}"
)

### 2.1 Perplexity

Perplexity on a held-out set of $N$ tokens:

$$PP = \exp\left(-\frac{1}{N}\sum_{i}\log P(w_i \mid \text{context})\right)$$

Lower perplexity means the model assigns higher probability to the held-out text.


In [ ]:
def sentence_log_prob(sentence, n):
    padded = pad(replace_unk(sentence), n)
    log_prob, count = 0.0, 0
    for i in range(n - 1, len(padded)):
        ngram = tuple(padded[i - n + 1 : i + 1])
        log_prob += math.log(laplace_prob(ngram, n))
        count += 1
    return log_prob, count


def perplexity(sentences, n):
    total_log_prob, total_tokens = 0.0, 0
    for s in sentences:
        lp, c = sentence_log_prob(s, n)
        total_log_prob += lp
        total_tokens += c
    return math.exp(-total_log_prob / total_tokens)

### 2.2 Evaluate all four models on development and test sets

In [ ]:
results = []
for n in [1, 2, 3, 4]:
    name = {1: "Unigram", 2: "Bigram", 3: "Trigram", 4: "Quadrigram"}[n]
    t0 = time.time()
    pp_dev = perplexity(dev_set, n)
    pp_test = perplexity(test_set, n)
    results.append((name, pp_dev, pp_test))
    print(f"{name} evaluated in {time.time() - t0:.1f}s")

print()
print(f"{'Model':<12}{'Dev Perplexity':>18}{'Test Perplexity':>18}")
print("-" * 48)
for name, pp_dev, pp_test in results:
    print(f"{name:<12}{pp_dev:>18.2f}{pp_test:>18.2f}")

### 2.3 Discussion of results

Under **Laplace smoothing**, perplexity typically **increases** as $n$ grows (unigram best, quadrigram worst). That is a known effect of add-one smoothing, not a bug:

- The number of possible contexts grows like $V^{n-1}$. For trigrams and quadrigrams, most development/test contexts are rare or unseen.
- Add-one then puts a large share of probability mass on the huge set of unseen n-grams (`+1` in the numerator, `+V` in the denominator). With a large Telugu vocabulary this penalty is severe.
- The unigram model has a single shared context, so counts are dense and smoothing distorts them less.

This is why methods such as add-$k$ with $k < 1$, interpolation, backoff, or Kneser–Ney are used in later work (Assignment 5 uses add-$k$ with $k=0.3$).


### 2.4 Example: next-word predictions and sentence probability

Candidates are taken from n-grams actually observed after the given context (plus a few frequent words), so we do not scan the entire vocabulary.


In [ ]:
def top_next_words(context_tuple, n, k=5):
    """k most probable next words given a fixed (n-1)-word context."""
    _, _, context_to_words = models[n]
    seen = set(context_to_words.get(context_tuple, set()))
    # Always consider a few high-frequency words so the ranking is not empty
    for (w,), _ in models[1][0].most_common(20):
        seen.add(w)
    seen.update(["<UNK>", "</s>"])

    candidates = [(w, laplace_prob(context_tuple + (w,), n)) for w in seen]
    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[:k]


# Frequent bigram context from training (skip <s> / </s>)
sample_context = None
for ngram, _ in models[2][0].most_common(50):
    if ngram[0] not in {"<s>", "</s>", "<UNK>"}:
        sample_context = (ngram[0],)
        break

print(f"Top 5 words after {sample_context} (bigram):")
for w, p in top_next_words(sample_context, 2):
    print(f"  {w:<20} P = {p:.8f}")

tri_ctx = None
for ngram, _ in models[3][0].most_common(80):
    if ngram[0] not in {"<s>", "</s>"} and ngram[1] not in {"<s>", "</s>"}:
        tri_ctx = ngram[:2]
        break

print(f"\nTop 5 words after {tri_ctx} (trigram):")
for w, p in top_next_words(tri_ctx, 3):
    print(f"  {w:<20} P = {p:.8f}")

sample = test_set[0]
print(f"\nSample test sentence: {' '.join(sample)}")
for n in [1, 2, 3, 4]:
    lp, c = sentence_log_prob(sample, n)
    name = {1: "Unigram", 2: "Bigram", 3: "Trigram", 4: "Quadrigram"}[n]
    print(f"  {name:<10} log P(sentence) = {lp:.2f}   ({c} tokens)")

## Summary

| Requirement | What this notebook does |
|---|---|
| Data | 1,000,000 tokenized Telugu sentences from Assignment 1 |
| Split | 1,000 test + 1,000 development + remaining training |
| Models | Unigram, bigram, trigram, quadrigram counts from training |
| Smoothing | Add-one / Laplace with vocabulary size $V$ |
| Evaluation | Perplexity on development and test sets for all four models |
